In [3]:
import pandas as pd
from pathlib import Path


# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = r"C:/Users/shantanu.shinde/Downloads/Weekly_Revenue_Master_Data.xlsx"
OUTPUT_FOLDER = r"C:/Users/shantanu.shinde/Downloads/output"


# ==========================================
# CREATE OUTPUT DIRECTORY
# ==========================================
Path(OUTPUT_FOLDER).mkdir(exist_ok=True)


# ==========================================
# READ FILE
# ==========================================
print("Reading master file...")

df = pd.read_excel(
    INPUT_FILE,
    engine="openpyxl"
)

print(f"Rows Loaded: {len(df):,}")


# ==========================================
# VALIDATE REQUIRED COLUMNS
# ==========================================
required_columns = [
    "Grandparent_Agency",
    "Revenue_USD",
    "Booked_Revenue_USD",
    "Pacing_Pct",
    "Advertiser",
    "Campaign_Name"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing Columns: {missing_columns}"
    )


# ==========================================
# PROCESS AGENCIES
# ==========================================
summary_data = []

grouped = df.groupby("Grandparent_Agency")

print(
    f"Found {df['Grandparent_Agency'].nunique()} Agencies"
)

for agency, agency_df in grouped:

    # --------------------------------------
    # Create Safe Filename
    # --------------------------------------
    safe_name = (
        agency.replace("/", "-")
              .replace("\\", "-")
              .replace(":", "")
              .replace("*", "")
              .replace("?", "")
              .replace('"', "")
              .replace("<", "")
              .replace(">", "")
              .replace("|", "")
    )

    # --------------------------------------
    # Save Agency File
    # --------------------------------------
    agency_file = (
        Path(OUTPUT_FOLDER)
        / f"{safe_name}.xlsx"
    )

    agency_df.to_excel(
        agency_file,
        index=False,
        engine="openpyxl"
    )

    # --------------------------------------
    # Agency Summary Metrics
    # --------------------------------------
    total_revenue = agency_df[
        "Revenue_USD"
    ].sum()

    total_booked = agency_df[
        "Booked_Revenue_USD"
    ].sum()

    avg_pacing = agency_df[
        "Pacing_Pct"
    ].mean()

    advertiser_count = agency_df[
        "Advertiser"
    ].nunique()

    campaign_count = agency_df[
        "Campaign_Name"
    ].nunique()

    summary_data.append({
        "Agency": agency,
        "Total_Revenue_USD": round(total_revenue, 2),
        "Total_Booked_USD": round(total_booked, 2),
        "Avg_Pacing_Pct": round(avg_pacing, 2),
        "Advertiser_Count": advertiser_count,
        "Campaign_Count": campaign_count,
        "Record_Count": len(agency_df)
    })

    print(
        f"✅ {agency} -> "
        f"{len(agency_df):,} records"
    )


# ==========================================
# CREATE SUMMARY FILE
# ==========================================
summary_df = pd.DataFrame(summary_data)

summary_df = summary_df.sort_values(
    by="Total_Revenue_USD",
    ascending=False
)

summary_file = (
    Path(OUTPUT_FOLDER)
    / "Agency_Summary.xlsx"
)

summary_df.to_excel(
    summary_file,
    index=False,
    engine="openpyxl"
)

print("\n--------------------------------")
print("Processing Complete")
print("--------------------------------")
print(
    f"Agency Files Generated : {len(summary_df)}"
)
print(
    f"Summary File Generated : {summary_file}"
)

Reading master file...
Rows Loaded: 468
Found 6 Agencies
✅ Dentsu -> 78 records
✅ Havas Media -> 78 records
✅ IPG Mediabrands -> 78 records
✅ Omnicom Media Group -> 78 records
✅ Publicis Groupe -> 78 records
✅ WPP Media -> 78 records

--------------------------------
Processing Complete
--------------------------------
Agency Files Generated : 6
Summary File Generated : C:\Users\shantanu.shinde\Downloads\output\Agency_Summary.xlsx
